In [10]:
import numpy as np
from firedrake import *
from firedrake.cython import dmcommon
from petsc4py import PETSc
import math

In [ ]:
# parameters in SI units
t_end = 5.0  # time of simulation [s]
dt = 0.005  # time step [s]
g = 9.8  # gravitational acceleration
# water
Lx = 20.0  # length of the tank [m] in x-direction; needed for computing initial condition
Lz = 10.0  # height of the tank [m]; needed for computing initial condition
rho = 1000.0  # fluid density in kg/m^2 in 2D [water]
# solid parameters
#  - we use a sufficiently soft material to be able to see noticeable structural displacement
rho_B = 7700.0  # structure density in kg/m^2 in 2D
lam = 1e7  # N/m in 2D - first Lame constant
mu = 1e7  # N/m in 2D - second Lame constant

# these numbers must match the ones defined in the mesh file
fluid_id = 1  # fluid subdomain
structure_id = 2  # structure subdomain
bottom_id = 1  # structure bottom
top_id = 6  # fluid surface
interface_id = 9  # fluid-structure interface

L = Lz
T = L / math.sqrt(g * L)
t_end /= T
dt /= T
Lx /= L
Lz /= L
rho_B /= rho
lam /= g * rho * L
mu /= g * rho * L
rho = 1.0  # or equivalently rho /= rho

mesh = Mesh("/home/charlottecai/MSc_Project_new/L_domain.msh")
dim = 2

# Extract submesh
mesh_F = Submesh(mesh, dim, fluid_id,label_name=dmcommon.CELL_SETS_LABEL, name="fluid_mesh")
x_F = SpatialCoordinate(mesh_F)
n_F = FacetNormal(mesh_F)

mesh_S = Submesh(mesh, dim, structure_id, label_name=dmcommon.CELL_SETS_LABEL, name="solid_mesh")
x_S = SpatialCoordinate(mesh_S)
n_S = FacetNormal(mesh_S)

# Function spaces
V_W = FunctionSpace(mesh_F, "CG", 1)
V_B = VectorFunctionSpace(mesh_S, "CG", 1)

mixed_V = V_W * V_B

# Measures
dx_F = Measure("dx", mesh_F)
dx_S = Measure("dx", mesh_S)

# interface measures
ds_F = Measure("ds", mesh_F, intersect_measures=(Measure("ds", mesh_S),))
dS_S = Measure("ds", mesh_S, intersect_measures=(Measure("ds", mesh_F),))

